# 🚢 Titanic — Modelo 1: Regressão Logística

Este notebook treina e avalia um modelo de Regressão Logística com GridSearchCV.

**Pipeline:**
```
ColumnDropper → CategorizerFeatures → OrdinalEncoder → OneHotEncoder → Imputer → Scaler → LogisticRegression
```

---
> **Fluxo:** `01_EDA` → **`02_Logistic_Regression`** → `03_Random_Forest` → `04_Voting_Classifier`

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from data_transform import build_preprocessing_pipeline

## 2. Carregamento dos dados

In [ ]:
df_train   = pd.read_csv('dataset/titanicc/train.csv')
df_test    = pd.read_csv('dataset/titanicc/test.csv')
df_labels  = pd.read_csv('dataset/titanicc/gender_submission.csv')

FEATURES = ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age',
            'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

X_train  = df_train[FEATURES]
y_train  = df_train['Survived']

X_test   = df_test
y_test   = df_labels['Survived'].to_numpy()

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

## 3. Pré-processamento + GridSearchCV

In [ ]:
ppl_transform = build_preprocessing_pipeline()
X_train_ppl   = ppl_transform.fit_transform(X_train)

param_grid = {
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear', 'saga'],
    'C'      : [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid_search.fit(X_train_ppl, y_train)

best_params = grid_search.best_params_
print("Melhores parâmetros :", best_params)
print(f"Melhor score (CV=5) : {grid_search.best_score_:.4f}")

## 4. Treino e avaliação no conjunto de teste

In [ ]:
model = make_pipeline(
    build_preprocessing_pipeline(),
    LogisticRegression(**best_params, max_iter=1000)
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Morreu', 'Sobreviveu']))

## 5. Matriz de Confusão

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Morreu', 'Sobreviveu'],
    cmap='Blues', ax=ax
)
ax.set_title('Regressão Logística — Matriz de Confusão', fontsize=13)
plt.tight_layout()
plt.show()

---
**Próximo passo:** `03_Random_Forest.ipynb`